In [0]:
pip install jinja2

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from jinja2 import Template

In [0]:
parameters = [
    {
        "table": "spotify_cata.silver.factstream",
        "alias": "factstream",
        "cols": "factstream.stream_id,factstream.listen_duration"
    },
    {
        "table": "spotify_cata.silver.dimuser",
        "alias": "dimuser",
        "cols": "dimuser.user_id,dimuser.user_name",
        "conditions": "factstream.user_id = dimuser.user_id"
    },
    {
        "table": "spotify_cata.silver.dimtrack",
        "alias": "dimtrack",
        "cols": "dimtrack.track_id",
        "conditions": "factstream.track_id = dimtrack.track_id"
    }
]

In [0]:
query_text = """
SELECT
    {% for param in parameters %}
        {{ param.cols }}
        {% if not loop.last %}
            ,
        {% endif %}
    {% endfor %}
FROM
    {% for param in parameters%}
        {%if loop.first%}
        {{param['table']}} AS {{ param.['alias']}}
        {% endif %}
        {% endfor %}
        {% for param in parameters%}
        {% if loop.first %}
        LEFT JOIN
            {{ param.table }} AS {{ param.alias }}
        ON
            {{ param.conditions }}
        {% endif %}
        {% endfor %}
"""


In [0]:
query_text = """
SELECT
    {% for param in parameters %}
        {{ param.cols }}
        {% if not loop.last %}
            ,
        {% endif %}
    {% endfor %}
FROM
    {{ parameters[0]['table'] }} AS {{ parameters[0]['alias'] }}
    {% for param in parameters[1:] %}
        LEFT JOIN {{ param['table'] }} AS {{ param['alias'] }}
        ON {{ param['conditions'] }}
    {% endfor %}
"""


In [0]:
jinja_sql_str = Template(query_text)
query = jinja_sql_str.render(parameters=parameters)
print(query)


SELECT
    
        factstream.stream_id,factstream.listen_duration
        
            ,
        
    
        dimuser.user_id,dimuser.user_name
        
            ,
        
    
        dimtrack.track_id
        
    
FROM
    spotify_cata.silver.factstream AS factstream
    
        LEFT JOIN spotify_cata.silver.dimuser AS dimuser
        ON factstream.user_id = dimuser.user_id
    
        LEFT JOIN spotify_cata.silver.dimtrack AS dimtrack
        ON factstream.track_id = dimtrack.track_id
    


In [0]:
spark.sql(query)
display(spark.sql(query))

stream_id,listen_duration,user_id,user_name,track_id
1,156,361,Joe Moore,74
2,47,321,Kyle Holden,288
3,214,275,Rachel York,340
4,14,43,David Taylor,373
5,266,319,Richard Chambers,95
6,317,52,Gabrielle Garza,31
7,90,5,Yolanda Morris,354
8,159,115,Chad Jones,386
9,290,439,Monica Barrera,95
10,53,40,Jacqueline Harrington,389
